In [ ]:
# ============================================================
# Nepal House Price Prediction — End to End
# Dataset: Hamrobazaar / Nepal Real Estate (2020-4-27.csv)
# Author: aiwithprakas.com.np
# ============================================================

# ── IMPORTS ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import re
import warnings
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — LOAD DATA
# ─────────────────────────────────────────────────────────────────────────────
df = pd.read_csv("./Dataset/2020-4-27.csv")
print(f"Raw shape: {df.shape}")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — FILTER: House For Sale only
# ─────────────────────────────────────────────────────────────────────────────
df = df[df['Title'].str.contains('House For Sale', case=False, na=False)].copy()
print(f"After House For Sale filter: {df.shape}")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — PARSE HELPER FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

# --- Views: "2.6K" → 2600 ---
def parse_views(v):
    v = str(v).strip().upper().replace(',', '')
    try:
        return float(v.replace('K', '')) * 1000 if 'K' in v else float(v)
    except:
        return np.nan

# --- Area: Aana/Ropani/Sqft → sqft ---
# Nepal land units:
#   1 Ropani = 16 Aana = 5476 sqft
#   1 Aana   = 4 Paisa = 342.25 sqft
#   1 Paisa  = 4 Dam   = 85.56 sqft
#   1 Dam    =           21.39 sqft
def parse_area(s):
    s = str(s).strip()
    # Normalize dot-separated numbers (e.g. "3.1.0 Aana" → "3-1-0 Aana")
    s2 = re.sub(r'(\d)\.(\d)', r'\1-\2', s)

    # R-A-P-D Aana (4 parts)
    m = re.match(r'^(\d+)-(\d+)-(\d+)-(\d+)\s*/?\s*[\d.]*\s*Aana', s2, re.I)
    if m:
        r, a, p, d = int(m.group(1)), int(m.group(2)), int(m.group(3)), int(m.group(4))
        return r * 16 * 342.25 + a * 342.25 + p * 85.56 + d * 21.39
    # R-A-P Aana (3 parts)
    m = re.match(r'^(\d+)-(\d+)-(\d+)\s*/?\s*[\d.]*\s*Aana', s2, re.I)
    if m:
        r, a, p = int(m.group(1)), int(m.group(2)), int(m.group(3))
        return r * 16 * 342.25 + a * 342.25 + p * 85.56
    # X Aana / X ana Aana
    m = re.match(r'^([\d.]+)\s*(?:ana\s*|aana\s*)?Aana', s, re.I)
    if m:
        try: return float(m.group(1)) * 342.25
        except: return np.nan
    # R-A-P-D Ropani
    m = re.match(r'^(\d+)-(\d+)-(\d+)-(\d+)\s*Ropani', s2, re.I)
    if m:
        r, a, p, d = int(m.group(1)), int(m.group(2)), int(m.group(3)), int(m.group(4))
        return r * 5476 + a * 342.25 + p * 85.56 + d * 21.39
    # X Ropani
    m = re.match(r'^([\d.]+)\s*Ropani', s, re.I)
    if m: return float(m.group(1)) * 5476
    # X Sq. Feet
    m = re.match(r'^([\d.]+)\s*Sq', s, re.I)
    if m: return float(m.group(1))
    return np.nan

# --- Road Width: "20 Feet / Blacktopped" → 20 ---
def parse_road_width(rw):
    m = re.search(r'([\d.]+)\s*(Feet|Meter|feet|meter)', str(rw))
    if m:
        val = float(m.group(1))
        if 'meter' in m.group(2).lower():
            val *= 3.281  # meter to feet
        return val
    return np.nan

# --- Road Type: string → rank ---
road_rank = {'Blacktopped': 4, 'Concrete': 3, 'Paved': 2, 'Gravelled': 1}
def parse_road_type(rt):
    for k, v in road_rank.items():
        if k.lower() in str(rt).lower():
            return v
    return 0

# --- Amenities: list string → count ---
def count_amenities(a):
    try: return len(eval(str(a)))
    except: return 0

# ─────────────────────────────────────────────────────────────────────────────
# STEP 4 — FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────────────────────
df['Views_num']       = df['Views'].apply(parse_views)
df['Area_sqft']       = df['Area'].apply(parse_area)
df['RoadWidth_ft']    = df['Road Width'].apply(parse_road_width)
df['RoadType_num']    = df['Road Type'].apply(parse_road_type)

# Fallback: if Road Type empty, get from Road column
mask = df['RoadType_num'] == 0
df.loc[mask, 'RoadType_num'] = df.loc[mask, 'Road'].apply(parse_road_type)

# Face direction (South/East = premium in Nepal)
face_map = {
    'East': 1, 'West': 2, 'North': 3, 'South': 4,
    'North East': 5, 'North West': 6, 'South East': 7, 'South West': 8
}
df['Face_num'] = df['Face'].map(face_map).fillna(0)

# City premium (Kathmandu > Lalitpur/Pokhara > others)
city_map = {'Kathmandu': 3, 'Lalitpur': 2, 'Pokhara': 2, 'Bhaktapur': 1}
df['City_num'] = df['City'].map(city_map).fillna(0)

df['Amenities_count'] = df['Amenities'].apply(count_amenities)
df['HasParking']      = (df['Parking'] > 0).astype(int)

# Price per sqft (proxy feature)
df['PricePerSqft_est'] = df['Price'] / (df['Area_sqft'] + 1)

print("\nNew features added: Area_sqft, RoadWidth_ft, RoadType_num, Face_num, City_num, Amenities_count, HasParking")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 5 — EDA
# ─────────────────────────────────────────────────────────────────────────────
print("\n── EDA ─────────────────────────────────────────────────")
print("Missing values:\n", df.isnull().sum()[df.isnull().sum() > 0])

# Price distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['Price'].dropna() / 1e6, bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Price (Million NPR) — Right Skewed')
axes[0].set_xlabel('Price (Million NPR)')

axes[1].hist(np.log1p(df['Price'].dropna()), bins=50, color='teal', edgecolor='white')
axes[1].set_title('Log(Price) — More Normal')
axes[1].set_xlabel('log(Price)')
plt.tight_layout()
plt.savefig('eda_price_distribution.png', dpi=100, bbox_inches='tight')
plt.close()
print("Saved: eda_price_distribution.png")
print(f"Price skewness (original): {df['Price'].skew():.2f}")
print(f"Price skewness (log):      {np.log1p(df['Price']).skew():.2f}")

# City-wise price
print("\nCity-wise median price (NPR):")
print(df.groupby('City')['Price'].median().sort_values(ascending=False).head(8).apply(lambda x: f"{x/1e6:.1f}M"))

# Correlation with price
num_df = df[['Price','Area_sqft','Bedroom','Bathroom','Floors','Parking',
             'RoadWidth_ft','Amenities_count','Views_num']].dropna()
corr = num_df.corr()['Price'].sort_values(ascending=False)
print("\nCorrelation with Price:")
print(corr.round(3))

# ─────────────────────────────────────────────────────────────────────────────
# STEP 6 — OUTLIER REMOVAL
# ─────────────────────────────────────────────────────────────────────────────
print("\n── Outlier Removal ─────────────────────────────────────")
print(f"Before: {df.shape[0]} rows")

# Price: 1M to 500M NPR (reasonable range)
df = df[(df['Price'] >= 1_000_000) & (df['Price'] <= 500_000_000)]

# Area: 50 to 50,000 sqft
df = df[(df['Area_sqft'] > 50) & (df['Area_sqft'] < 50_000)]

# Bedroom/Bathroom extreme outliers
df = df[(df['Bedroom'] <= 15) & (df['Bathroom'] <= 15)]

print(f"After:  {df.shape[0]} rows")

# Scatter: Area vs Price
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df['Area_sqft'], df['Price'] / 1e6, alpha=0.4, s=20, color='coral')
ax.set_xlabel('Area (sqft)')
ax.set_ylabel('Price (Million NPR)')
ax.set_title('Area vs Price — Nepal Housing')
plt.savefig('eda_area_vs_price.png', dpi=100, bbox_inches='tight')
plt.close()
print("Saved: eda_area_vs_price.png")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 7 — FINAL FEATURES & CLEAN
# ─────────────────────────────────────────────────────────────────────────────
features = [
    'Bedroom', 'Bathroom', 'Floors', 'Parking',
    'Area_sqft', 'RoadWidth_ft', 'RoadType_num',
    'Face_num', 'City_num', 'Amenities_count', 'HasParking', 'Views_num'
]

df_clean = df[features + ['Price']].copy()

# Fill remaining nulls
df_clean['Floors']       = df_clean['Floors'].fillna(df_clean['Floors'].median())
df_clean['RoadWidth_ft'] = df_clean['RoadWidth_ft'].fillna(df_clean['RoadWidth_ft'].median())
df_clean['Views_num']    = df_clean['Views_num'].fillna(df_clean['Views_num'].median())
df_clean = df_clean.dropna()

print(f"\nFinal clean dataset: {df_clean.shape}")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 8 — TRAIN / TEST SPLIT
# ─────────────────────────────────────────────────────────────────────────────
X = df_clean[features].copy()
y = np.log1p(df_clean['Price'])   # log transform target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"\nTrain: {X_train.shape}, Test: {X_test.shape}")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 9 — MODEL TRAINING
# ─────────────────────────────────────────────────────────────────────────────
print("\n── Model Training ──────────────────────────────────────")

def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    r2        = r2_score(y_te, pred)
    rmse_log  = np.sqrt(mean_squared_error(y_te, pred))
    rmse_npr  = np.sqrt(mean_squared_error(np.expm1(y_te), np.expm1(pred)))
    mae_npr   = mean_absolute_error(np.expm1(y_te), np.expm1(pred))
    cv_scores = cross_val_score(model, X_tr, y_tr, cv=5, scoring='r2')
    print(f"\n{name}:")
    print(f"  R² Score:       {r2:.4f}")
    print(f"  CV R² (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    print(f"  RMSE (log):     {rmse_log:.4f}")
    print(f"  RMSE (NPR):     NPR {rmse_npr:,.0f}")
    print(f"  MAE  (NPR):     NPR {mae_npr:,.0f}")
    return model

# Baseline
ridge = evaluate_model(
    "Ridge Regression (baseline)",
    Ridge(alpha=10),
    X_train, X_test, y_train, y_test
)

# Best model
xgb = evaluate_model(
    "XGBoost",
    XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=3,
        reg_alpha=0.1,
        random_state=42,
        verbosity=0
    ),
    X_train, X_test, y_train, y_test
)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 10 — FEATURE IMPORTANCE
# ─────────────────────────────────────────────────────────────────────────────
fi = pd.Series(xgb.feature_importances_, index=features).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8, 6))
fi.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Feature Importance — XGBoost')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=100, bbox_inches='tight')
plt.close()
print("\nFeature Importance (top → bottom):")
print(fi.sort_values(ascending=False).round(3))

# ─────────────────────────────────────────────────────────────────────────────
# STEP 11 — RESIDUAL PLOT
# ─────────────────────────────────────────────────────────────────────────────
pred_test = xgb.predict(X_test)
residuals = y_test - pred_test

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(pred_test, residuals, alpha=0.4, s=20, color='coral')
axes[0].axhline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_xlabel('Predicted (log scale)')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residual Plot — Random scatter = Good!')

axes[1].scatter(np.expm1(y_test)/1e6, np.expm1(pred_test)/1e6, alpha=0.4, s=20, color='teal')
axes[1].plot([0, 500], [0, 500], 'r--', linewidth=1)
axes[1].set_xlabel('Actual Price (M NPR)')
axes[1].set_ylabel('Predicted Price (M NPR)')
axes[1].set_title('Actual vs Predicted')
plt.tight_layout()
plt.savefig('residual_plots.png', dpi=100, bbox_inches='tight')
plt.close()
print("\nSaved: residual_plots.png")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 12 — SKLEARN PIPELINE (for deployment)
# ─────────────────────────────────────────────────────────────────────────────
print("\n── Building Deployment Pipeline ────────────────────────")

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, features)
], remainder='drop')

full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=3,
        reg_alpha=0.1,
        random_state=42,
        verbosity=0
    ))
])

# Train on full data (all X, y)
full_pipeline.fit(X, y)

# Test pipeline prediction
sample = pd.DataFrame([{
    'Bedroom': 5, 'Bathroom': 3, 'Floors': 3.0, 'Parking': 1,
    'Area_sqft': 1368.0,   # 4 Aana
    'RoadWidth_ft': 20.0, 'RoadType_num': 4,  # blacktopped
    'Face_num': 1,          # East
    'City_num': 3,          # Kathmandu
    'Amenities_count': 6, 'HasParking': 1, 'Views_num': 500.0
}])

log_pred = full_pipeline.predict(sample)[0]
price_pred = np.expm1(log_pred)
print(f"\nSample prediction: NPR {price_pred:,.0f} ({price_pred/1e7:.2f} Crore)")

# Save pipeline
import os
os.makedirs('model', exist_ok=True)
with open('model/pipeline.pkl', 'wb') as f:
    pickle.dump(full_pipeline, f)

# Save feature names for Django
with open('model/features.pkl', 'wb') as f:
    pickle.dump(features, f)

print("\nPipeline saved → model/pipeline.pkl")
print("Features saved → model/features.pkl")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 13 — PREDICTION FUNCTION (use in Django views.py)
# ─────────────────────────────────────────────────────────────────────────────
def predict_price(bedroom, bathroom, floors, parking_spots,
                  area_aana, road_width_ft, road_type,
                  face, city, amenities_count):
    """
    area_aana   : float  (e.g. 4.0 for 4 Aana)
    road_type   : str    ('Blacktopped','Gravelled','Concrete','Paved')
    face        : str    ('East','West','North','South',...)
    city        : str    ('Kathmandu','Lalitpur','Bhaktapur','Pokhara')
    """
    area_sqft = area_aana * 342.25
    rt_map    = {'Blacktopped':4,'Concrete':3,'Paved':2,'Gravelled':1}
    face_map_ = {'East':1,'West':2,'North':3,'South':4,
                 'North East':5,'North West':6,'South East':7,'South West':8}
    city_map_ = {'Kathmandu':3,'Lalitpur':2,'Pokhara':2,'Bhaktapur':1}

    data = pd.DataFrame([{
        'Bedroom': bedroom, 'Bathroom': bathroom,
        'Floors': floors, 'Parking': parking_spots,
        'Area_sqft': area_sqft, 'RoadWidth_ft': road_width_ft,
        'RoadType_num': rt_map.get(road_type, 0),
        'Face_num': face_map_.get(face, 0),
        'City_num': city_map_.get(city, 0),
        'Amenities_count': amenities_count,
        'HasParking': int(parking_spots > 0),
        'Views_num': 500.0
    }])

    log_p = full_pipeline.predict(data)[0]
    return np.expm1(log_p)

# Test
price = predict_price(
    bedroom=5, bathroom=3, floors=3, parking_spots=2,
    area_aana=4.0, road_width_ft=20, road_type='Blacktopped',
    face='East', city='Kathmandu', amenities_count=8
)
print(f"\nTest predict (5bed, 3bath, 4Aana, KTM): NPR {price:,.0f}")

print("\n✓ All done! Check model/ folder for pipeline.pkl")